# Audit di bias su Google Shopping — Learning-to-rank (versione piena, narrata)

**Obiettivo.** Capire se il ranking di Google Shopping favorisce certi venditori **oltre** il merito
osservabile. Ricostruiamo l'ordine di Google con **LightGBM/LambdaMART** raggruppato per SERP
(`run_id`), valutiamo con **NDCG@10** su SERP held-out e interpretiamo con **SHAP**.

Confrontiamo due insiemi di feature:
- **solo merito** — prezzo, rilevanza keyword↔titolo, lunghezza titolo, categoria, branded/generic;
- **merito + venditore** — aggiunge `is_amazon`, `is_giant`, `seller_freq_log`.

Se il secondo ricostruisce l'ordine **meglio**, il venditore porta informazione oltre il merito
misurabile → *candidato* segnale di bias. Lo SHAP dà la **direzione** (Amazon spinto su o giù?).

> ⚠️ Stima **associativa**, non causale. Parte del lift può essere merito non misurato (rating,
> sponsorizzato, rilevanza vera) che si "rifugia" nell'identità del venditore. E ricorda i buchi nei
> dati: `rating`/`reviews_count` vuoti al ~98,6% e **nessun flag sponsorizzato**.

## 0. Test GPU
Il MiniLM gira anche su CPU, ma con GPU l'encoding è molto più veloce. Verifica che torch veda la
scheda (serve il build `cu124` di torch su macchina NVIDIA).

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("CUDA disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
    y = (torch.rand(5000,5000,device="cuda") @ torch.rand(5000,5000,device="cuda")); torch.cuda.synchronize()
    print("smoke test matmul GPU OK:", tuple(y.shape))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device scelto:", DEVICE)

## 1. Config
- `ENCODER="sentence-transformers"` usa il **MiniLM pieno**.
- `ST_LOCAL_PATH`: se valorizzato (cartella `minilm_it/` scompattata), carica il modello **offline**
  senza scaricarlo da Hugging Face. Lascialo vuoto per scaricarlo dall'hub al primo run (~470 MB).

In [ ]:
DB_PATH = "results.db"
COUNTRY = "IT"                       # 'IT' | 'DE' | 'EN'
ENCODER = "sentence-transformers"    # 'sentence-transformers' | 'model2vec' | 'tfidf'
N_SEEDS = 10
ST_LOCAL_PATH = ""                   # es. "minilm_it"  -> carica offline; "" -> scarica da HF

ST_MODEL  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
M2V_MODEL = "minishlab/potion-multilingual-128M"

import warnings, sqlite3; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb, shap
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score

## 2. Rilevanza keyword↔titolo
Tre encoder intercambiabili. Quello "vero" è il **MiniLM** (semantico): coglie i sinonimi
(*"scarpe running"* ↔ *"scarpa da corsa"*) che il TF-IDF lessicale manca. Per efficienza codifichiamo
solo le **stringhe uniche** (le keyword si ripetono molto tra le ~1.500 SERP). Fallback automatico a
TF-IDF se l'encoder non è disponibile.

In [ ]:
def _cosine_rows(K, T):
    K = K/(np.linalg.norm(K,axis=1,keepdims=True)+1e-9); T = T/(np.linalg.norm(T,axis=1,keepdims=True)+1e-9)
    return (K*T).sum(1)
def _encode_unique(enc, texts):
    uniq = pd.Index(texts.fillna("").unique()); emb = np.asarray(enc(uniq.tolist()))
    return emb[uniq.get_indexer(texts.fillna(""))]

def rel_sentence_transformers(df):
    from sentence_transformers import SentenceTransformer
    src = ST_LOCAL_PATH or ST_MODEL
    m = SentenceTransformer(src, device=DEVICE, **({"local_files_only": True} if ST_LOCAL_PATH else {}))
    enc = lambda xs: m.encode(xs, normalize_embeddings=True, show_progress_bar=False, batch_size=256)
    return (_encode_unique(enc, df["keyword"]) * _encode_unique(enc, df["title"])).sum(1)

def rel_model2vec(df):
    from model2vec import StaticModel
    m = StaticModel.from_pretrained(M2V_MODEL); enc = lambda xs: m.encode(xs, show_progress_bar=False)
    return _cosine_rows(_encode_unique(enc, df["keyword"]), _encode_unique(enc, df["title"]))

def rel_tfidf(df):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2, max_features=20000)
    M = vec.fit_transform(pd.concat([df["keyword"], df["title"].fillna("")])); n=len(df); K,T=M[:n],M[n:]
    kn=np.sqrt(np.asarray(K.multiply(K).sum(1)).ravel()); tn=np.sqrt(np.asarray(T.multiply(T).sum(1)).ravel())
    return np.asarray(K.multiply(T).sum(1)).ravel()/np.where(kn*tn==0,1,kn*tn)

def compute_relevance(df, encoder):
    fn={"sentence-transformers":rel_sentence_transformers,"model2vec":rel_model2vec,"tfidf":rel_tfidf}[encoder]
    try: return fn(df), encoder
    except Exception as e:
        print(f"[rel] '{encoder}' non disponibile ({type(e).__name__}: {e}) -> fallback TF-IDF")
        return rel_tfidf(df), "tfidf(fallback)"

## 3. Dati e feature
Carichiamo il mercato scelto, teniamo solo i prezzi validi, calcoliamo la rilevanza e costruiamo le
feature. L'etichetta `y` è graduata dalla posizione (top‑2 → 4, …, oltre la 20ª → 0).

In [ ]:
con=sqlite3.connect(DB_PATH); df=pd.read_sql("SELECT * FROM products WHERE language=?",con,params=(COUNTRY,)); con.close()
df=df[df["price_value"].notna() & (df["price_value"]>0)].reset_index(drop=True)
print(f"righe {len(df):,} | SERP {df['run_id'].nunique()}")

rel, used = compute_relevance(df, ENCODER); df["rel"]=rel
print(f"rilevanza: {used} | mean={rel.mean():.3f} std={rel.std():.3f}")

t=df["title"].fillna("")
df["log_price"]=np.log1p(df["price_value"]); df["title_len"]=t.str.len(); df["title_words"]=t.str.split().apply(len)
df["is_branded"]=(df["query_type"]=="branded").astype(int)
df["is_amazon"]=df["seller"].fillna("").str.contains("amazon",case=False).astype(int)
df["is_giant"]=df["seller"].isin(df["seller"].value_counts().head(15).index).astype(int)
df["seller_freq_log"]=np.log1p(df["seller"].map(df["seller"].value_counts()).fillna(0))
for c in ("category_l1","category_l2"): df[c]=df[c].astype("category")
df["y"]=df["position"].apply(lambda p:4 if p<=2 else 3 if p<=5 else 2 if p<=10 else 1 if p<=20 else 0)

MERIT=["log_price","rel","title_len","title_words","is_branded","category_l1","category_l2"]
PLATFORM=["is_amazon","is_giant","seller_freq_log"]; CAT=["category_l1","category_l2"]

## 4. Baseline e modelli (singolo split)
Due baseline di riferimento — ordine **casuale** e ordine per **prezzo crescente** — più i due modelli.
Tutto valutato sulle **stesse SERP di test** per essere confrontabile.

In [ ]:
def split(df, seed=42):
    tr,te=next(GroupShuffleSplit(1,test_size=0.3,random_state=seed).split(df,groups=df["run_id"])); return tr,te
def grp(d): return d.groupby("run_id",sort=False).size().values

def ndcg_baseline(dte, score):
    tot=0.0; k=0
    for _,g in dte.groupby("run_id",sort=False):
        if len(g)<2: continue
        tot += ndcg_score([g["y"].values],[score(g)],k=10); k+=1
    return tot/k

def train(df, feats, tr, te):
    dtr=df.iloc[tr].sort_values("run_id"); dte=df.iloc[te].sort_values("run_id")
    m=lgb.LGBMRanker(objective="lambdarank",metric="ndcg",n_estimators=300,learning_rate=0.05,num_leaves=31,
        min_child_samples=30,subsample=0.8,colsample_bytree=0.8,random_state=0,n_jobs=2,verbose=-1,label_gain=[0,1,3,7,15])
    m.fit(dtr[feats],dtr["y"],group=grp(dtr),eval_set=[(dte[feats],dte["y"])],eval_group=[grp(dte)],
          eval_at=[10],categorical_feature=[c for c in CAT if c in feats])
    return m, dte, m.best_score_["valid_0"]["ndcg@10"]

tr,te=split(df,42); dte_all=df.iloc[te].sort_values("run_id")
rng=np.random.default_rng(0)
nd_rand  = ndcg_baseline(dte_all, lambda g: rng.random(len(g)))
nd_price = ndcg_baseline(dte_all, lambda g: -g["price_value"].values)   # prezzo crescente
mA,_,nA  = train(df, MERIT, tr, te)
mB,dte,nB= train(df, MERIT+PLATFORM, tr, te)
results={"random":nd_rand,"prezzo crescente":nd_price,"solo merito":nA,"merito + venditore":nB}
for k,v in results.items(): print(f"  {k:<20} NDCG@10={v:.3f}")
print(f"\nLIFT venditore: {nB-nA:+.3f}")

### Grafico 1 — NDCG@10 per modello
Quanto bene ciascuna strategia ricostruisce l'ordine di Google. Il salto da *solo merito* a
*merito+venditore* è il **lift**.

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.6))
ks=list(results); vs=[results[k] for k in ks]
cols=["#bdbdbd","#bdbdbd","#4c78a8","#e45756"]
b=ax.barh(ks, vs, color=cols); ax.invert_yaxis(); ax.set_xlim(0.35, max(vs)+0.03)
for r,v in zip(b,vs): ax.text(v+0.003, r.get_y()+r.get_height()/2, f"{v:.3f}", va="center", fontsize=10)
ax.set_xlabel("NDCG@10 (SERP di test)"); ax.set_title(f"Ricostruzione dell'ordine di Google — {COUNTRY} ({used})")
ax.annotate(f"lift {nB-nA:+.3f}", xy=(nB,3), xytext=(nA,3.35),
            arrowprops=dict(arrowstyle="->"), fontsize=9, color="#e45756")
plt.tight_layout(); plt.savefig("nb_ndcg.png",dpi=120,bbox_inches="tight"); plt.show()

## 5. Stabilità su più split
Un solo split SERP è rumoroso: ripetiamo su `N_SEEDS` partizioni diverse e guardiamo media ± deviazione
del lift e dello SHAP di `is_amazon`.

In [ ]:
rows=[]
for s in range(N_SEEDS):
    tr,te=split(df,s)
    _,_,a=train(df,MERIT,tr,te); mB,dte_s,b=train(df,MERIT+PLATFORM,tr,te)
    sv=shap.TreeExplainer(mB).shap_values(dte_s[MERIT+PLATFORM])
    ia=dte_s["is_amazon"].values; sa=sv[:,(MERIT+PLATFORM).index("is_amazon")][ia==1].mean()
    rows.append((a,b,b-a,sa))
A=np.array(rows)
print(f"encoder={used} | {N_SEEDS} seed")
print(f"  solo merito : {A[:,0].mean():.3f} ± {A[:,0].std():.3f}")
print(f"  +venditore  : {A[:,1].mean():.3f} ± {A[:,1].std():.3f}")
print(f"  lift        : {A[:,2].mean():+.3f} ± {A[:,2].std():.3f}")
print(f"  is_amazon   : {A[:,3].mean():+.3f} ± {A[:,3].std():.3f}")

### Grafico 2 — Distribuzione del lift sui seed
Se la nuvola di punti sta tutta sopra lo zero, il lift è **robusto** (il venditore aiuta sempre).

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.2))
ax.axvline(0,color="#999",lw=1,ls="--")
ax.scatter(A[:,2], np.zeros(len(A))+rng.normal(0,0.03,len(A)), color="#e45756", alpha=0.8, zorder=3)
ax.errorbar(A[:,2].mean(),0, xerr=A[:,2].std(), fmt="o", color="black", capsize=4, zorder=4,
            label=f"media {A[:,2].mean():+.3f} ± {A[:,2].std():.3f}")
ax.set_yticks([]); ax.set_xlabel("lift NDCG@10 (merito+venditore − solo merito)")
ax.set_title("Lift del venditore su {} split".format(N_SEEDS)); ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.savefig("nb_lift.png",dpi=120,bbox_inches="tight"); plt.show()

### Grafico 3 — SHAP del modello merito+venditore
Direzione e forza di ciascuna feature. Guarda `is_amazon`: se i punti `amazon=1` stanno a SHAP
**negativo**, Amazon è spinto **in basso** (nessun favoritismo). `seller_freq_log` (prevalenza) è di
solito la feature più forte tra quelle di piattaforma.

In [ ]:
sv=shap.TreeExplainer(mB).shap_values(dte[MERIT+PLATFORM])
shap.summary_plot(sv, dte[MERIT+PLATFORM], show=False, max_display=10)
plt.tight_layout(); plt.savefig("nb_shap.png",dpi=120,bbox_inches="tight"); plt.show()

## 6. Tabella riassuntiva → CSV

In [ ]:
summary=pd.DataFrame({
 "metric":["NDCG random","NDCG prezzo","NDCG solo merito","NDCG +venditore",
           "lift (mean)","lift (std)","is_amazon SHAP (mean)","is_amazon SHAP (std)"],
 "value":[nd_rand,nd_price,A[:,0].mean(),A[:,1].mean(),A[:,2].mean(),A[:,2].std(),A[:,3].mean(),A[:,3].std()],
 "country":COUNTRY,"encoder":used,"n_seeds":N_SEEDS})
summary.to_csv(f"bias_summary_{COUNTRY}_{used}.csv", index=False)
print(summary.to_string(index=False))

## Come leggere i risultati
- **Lift > 0 e stabile** → il venditore porta informazione oltre il merito → *candidato* bias (non causale).
- **`is_amazon` SHAP negativo** → nessun favoritismo pro-Amazon (anzi, penalizzato).
- **Confronto encoder**: l'ipotesi era che la rilevanza **semantica** vera alzasse il merito-only e
  **riducesse** il lift. Rilancia il notebook con `ENCODER="tfidf"` e confronta: se il lift **non** scende,
  è guidato dalla **prevalenza** del venditore (`seller_freq_log`), non da rilevanza non misurata.

**Limiti strutturali:** senza `rating` e senza **flag sponsorizzato**, il test "qualità-aggiustato" e la
misura diretta del bias a pagamento restano impossibili con questi dati.